# Milestone 1.5: Single-Compound ADMET Summary Report
Generate a structured safety and drug-likeness profile for a single compound.
Defines the API response schema used by the production backend.

In [ ]:
import os, re, glob, json, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import SVG, HTML, display
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from captum.attr import LayerIntegratedGradients
warnings.filterwarnings('ignore')

# ── Constants (same as M2, M3, M4) ─────────────────────────────────────────
BASE_MODEL = 'DeepChem/ChemBERTa-77M-MTR'
checkpoints = sorted(glob.glob('chemberta-tox21-multitarget-*'))
MODEL_DIR = checkpoints[-1]
NUM_TARGETS = 12
TARGET_NAMES = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
]
TARGET_IDX = {name: i for i, name in enumerate(TARGET_NAMES)}
THRESHOLDS = {
    'NR-AR': 0.95, 'NR-AR-LBD': 0.95, 'NR-AhR': 0.75,
    'NR-Aromatase': 0.85, 'NR-ER': 0.70, 'NR-ER-LBD': 0.85,
    'NR-PPAR-gamma': 0.85, 'SR-ARE': 0.65, 'SR-ATAD5': 0.80,
    'SR-HSE': 0.85, 'SR-MMP': 0.85, 'SR-p53': 0.85,
}
SEVERITY_WEIGHTS = {
    'NR-AR': 1.0, 'NR-AR-LBD': 1.0, 'NR-AhR': 1.5,
    'NR-Aromatase': 1.0, 'NR-ER': 1.0, 'NR-ER-LBD': 1.0,
    'NR-PPAR-gamma': 1.0, 'SR-ARE': 1.0, 'SR-ATAD5': 1.0,
    'SR-HSE': 1.0, 'SR-MMP': 1.5, 'SR-p53': 1.5,
}
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
WEIGHT_ARRAY = np.array([SEVERITY_WEIGHTS[t] for t in TARGET_NAMES])
WEIGHT_SUM   = WEIGHT_ARRAY.sum()
print(f"Checkpoint: {MODEL_DIR}  Device: {DEVICE}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=NUM_TARGETS,
    ignore_mismatched_sizes=True, attn_implementation='eager',
)
model = PeftModel.from_pretrained(base, MODEL_DIR).merge_and_unload()
model.eval().to(DEVICE)

def _forward_for_ig(input_ids, attention_mask, target_idx):
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    return torch.sigmoid(out.logits[:, target_idx])

lig = LayerIntegratedGradients(_forward_for_ig, model.roberta.embeddings)
print("Model and IG ready.")

In [ ]:
_ATOM_RE = re.compile(r'\[[^\]]+\]|Cl|Br|[BCNOPSFI]|[bcnops]')

_pains_params = FilterCatalogParams()
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
_pains_catalog = FilterCatalog(_pains_params)

def mol_to_svg(mol, width=360, height=260):
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    drawer.drawOptions().addStereoAnnotation = False
    drawer.DrawMolecule(mol)
    finish = getattr(drawer, 'FinishDrawing', None) or getattr(drawer, 'EndDrawing')
    finish()
    return drawer.GetDrawingText()

def compute_admet(mol):
    mw  = Descriptors.ExactMolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = Descriptors.NumHDonors(mol)
    hba  = Descriptors.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rb   = Descriptors.NumRotatableBonds(mol)
    violations = int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10)
    pains = [e.GetDescription() for e in _pains_catalog.GetMatches(mol)]
    return {
        "molecular_weight": round(mw, 2),
        "logp": round(logp, 2),
        "hbd": hbd, "hba": hba,
        "tpsa": round(tpsa, 2),
        "rotatable_bonds": rb,
        "lipinski_pass": violations <= 1,
        "veber_pass": tpsa <= 140 and rb <= 10,
        "pains_alerts": pains,
    }

In [ ]:
# === ASSERTION CELL ===
# Aspirin: MW=180, logP=1.2, HBD=1, HBA=3 — should pass Lipinski and Veber
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
admet_asp = compute_admet(aspirin)
assert admet_asp['lipinski_pass'], "Aspirin should pass Lipinski"
assert admet_asp['veber_pass'],    "Aspirin should pass Veber"
assert admet_asp['molecular_weight'] < 200, f"MW wrong: {admet_asp['molecular_weight']}"
assert admet_asp['pains_alerts'] == [], f"Aspirin should have no PAINS alerts"

# Cyclosporin A: MW~1202, should fail Lipinski
cyclosporin = Chem.MolFromSmiles(
    'CCC1C(=O)N(CC(=O)N(C(C(=O)NC(C(=O)N(C(C(=O)NC(C(=O)NC(C(=O)N(C(C(=O)N(C(C(=O)N(C(C(=O)N1C)CC(C)C)C)CC(C)C)C)C)CC(C)C)C)C(C)C)C)CC(C)C)C)C(C)C)C)C'
)
if cyclosporin:
    admet_cyc = compute_admet(cyclosporin)
    assert not admet_cyc['lipinski_pass'], "Cyclosporin should fail Lipinski (MW > 500)"

print("✓ ADMET assertions passed")
print(f"  Aspirin: MW={admet_asp['molecular_weight']}, logP={admet_asp['logp']}, "
      f"Lipinski={admet_asp['lipinski_pass']}, Veber={admet_asp['veber_pass']}")